# Physics Activity: Non-linear Pendulum & Phase Space Dynamics

In this activity, we will learn the physics of non-linear pendulums using simple Python code. Please complete the **Commenting Exercise** meant to practice the ability to read code and understand its purpose, and the **Student Exploration Worksheet** to understand differences between linear and non-linear pendulum behavior.

You will investigate the motion of a simple gravity pendulum without using the small-angle approximation $\sin(\theta) \approx \theta$.

### Equations of Motion:
* **True Non-linear Pendulum:**
  $$\frac{d^2\theta}{dt^2} + \gamma \frac{d\theta}{dt} + \frac{g}{L}\sin(\theta) = 0$$
* **Linear (Small-Angle) Approximation:**
  $$\frac{d^2\theta}{dt^2} + \gamma \frac{d\theta}{dt} + \frac{g}{L}\theta = 0$$

Run the code cells below to set up the simulation engine and launch the interactive sliders.

### Importing Python packages:
We start by importing the required packages; we use NumPy for numerical functions. Matplotlib is used to plot relationships between different quantities. We use the solve_ivp function from the SciPy package to solve differential equations. ipywidgets is used for interactive UI.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from ipywidgets import FloatSlider, Checkbox, VBox, HBox

### Defining functions
When writing a program, break its operations into separate Python functions. This approach helps organize long programs by isolating specific tasks within individual functions. If a bug occurs, it is more likely to be contained within a single function, making the program easier to test, debug, and maintain. Plan and define these functions before you start writing the program. Here, we break them down into five different functions.

1. Differential equation for linear pendulum
2. Differential equation for non-linear pendulum
3. Numerical solver function
4. Static Plotting function
5. Interactive plotting function

You can define with **def**. We start by defining a differential equation function for the linear pendulum. **t** is variable for time in seconds, **y** is State vector **theta**, angular displacement, and **omega**, angular velocity **d(theta)/dt** (rad/s), **g** is Gravitational acceleration (m/s^2), **length** is Pendulum rod length (m) and **damping** is Damping coefficient measured in s^-1.

In [2]:
def linear_pendulum_ode(t, y, g, length, damping):

    """
    Defines the system of first-order ODEs for a damped linear pendulum.

    Parameters:
    - t: Current time step (s)
    - y: State vector [theta, omega] where:
         theta = angular displacement (radians)
         omega = angular velocity d(theta)/dt (rad/s)
    - g: Gravitational acceleration (m/s^2)
    - length: Pendulum rod length (m)
    - damping: Damping coefficient measured (s^-1)

    Returns:
    - [dtheta/dt, domega/dt]
    """
    theta, omega = y
    dtheta_dt = omega
    domega_dt = -(g / length) * theta - damping * omega
    return [dtheta_dt, domega_dt]

Note that I have detailed comments within the function, indicated by blocks within """ and """, explaining what each variable is. Projects run for months or years and are shared with colleagues who build on your work. Commenting is crucial for code sharing and also for your own reference. Commenting can seem time-consuming, but it's important.

In [3]:
def nonlinear_pendulum_ode(t, y, g, length, damping):
    """
    Defines the exact non-linear pendulum system of first-order differential equations.
    
    Mathematical Equation:
        d²θ/dt² + γ*(dθ/dt) + (g/L)*sin(θ) = 0
    
    State Vector Reduction (2-state system):
        Let y[0] = θ (Angular displacement in radians)
        Let y[1] = ω = dθ/dt (Angular velocity in rad/s)
        
        Derivatives:
        dy[0]/dt = ω = y[1]
        dy[1]/dt = -(g/L)*sin(y[0]) - γ*y[1]
    
    Parameters:
    ----------
    t : float
        Current time step (s). Required by SciPy IVP solvers even if autonomous.
    y : list or ndarray of float
        Current state vector [theta, omega].
    g : float
        Acceleration due to gravity (m/s²).
    length : float
        Length of the pendulum rod L (m).
    damping : float
        Viscous friction coefficient γ (s⁻¹).
        
    Returns:
    -------
    list of float
        Time derivatives of state vector: [dtheta_dt, domega_dt].
    """
    # Unpack state vector variables
    theta = y[0]  # Current angle (radians)
    omega = y[1]  # Current angular velocity (radians/second)
    
    # 1. First-order derivative of position is angular velocity
    dtheta_dt = omega
    
    # 2. Acceleration derived from torque balance using EXACT sin(theta) restoring force
    domega_dt = -(g / length) * np.sin(theta) - (damping * omega)
    
    # Return rate of change vector matching input state shape
    return [dtheta_dt, domega_dt]

To solve the differential equation, we use the solver **solve_ivp**. I recommend starting by reviewing the documentation for the solver you want to use. It gives you an idea of what quantities you need for the solver, and then define those variables, such as **theta0_rad**, **t_span** and **t_eval** below. Then decide the variables for the function, **solve_pendulum_systems**, that you wrap around the solver.

In [4]:
def solve_pendulum_systems(theta_deg=30.0, omega0=0.0, length=1.0, damping=0.0, t_max=20.0, num_points=1000):
    """
    Numerically integrates both non-linear and linear pendulum initial value problems (IVP).
    
    Parameters:
    ----------
    theta_deg : float
        Initial angular offset in degrees (converted internally to radians).
    omega0 : float
        Initial angular velocity (rad/s).
    length : float
        Rod length L (m).
    damping : float
        Damping coefficient γ (s⁻¹).
    t_max : float
        Total duration of simulation in seconds.
    num_points : int
        Number of time steps to evaluate for smooth plotting.
        
    Returns:
    -------
    tuple (sol_nl, sol_lin)
        SciPy 'OdeResult' objects containing time arrays (.t) and solution trajectories (.y).
    """
    # Physical constants
    g = 9.81  # Gravitational acceleration (m/s^2)
    
    # Convert initial angle input from student-friendly degrees to standard SI radians
    theta0_rad = np.radians(theta_deg)
    
    # Define time domain boundaries: (start_time, end_time)
    t_span = (0.0, t_max)
    
    # Generate array of evenly spaced time points where solution values are recorded
    t_eval = np.linspace(0.0, t_max, num_points)
    
    # Combine initial position and velocity into starting state vector y0
    initial_state = [theta0_rad, omega0]

    # Integrate the non-linear system using explicit Runge-Kutta 4th/5th order (RK45)
    sol_nl = solve_ivp(
        fun=nonlinear_pendulum_ode,     # Target ODE function
        t_span=t_span,                  # Time interval tuple
        y0=initial_state,               # Initial state vector [theta0, omega0]
        args=(g, length, damping),      # Extra constant arguments passed to ODE function
        t_eval=t_eval,                  # Fixed evaluation timestamps
        method='RK45',                  # Adaptive step-size integrator
        rtol=1e-6,                      # Relative tolerance threshold for accuracy
        atol=1e-8                       # Absolute tolerance threshold
    )

    # Integrate the linear system under identical conditions for comparison
    sol_lin = solve_ivp(
        fun=linear_pendulum_ode,
        t_span=t_span,
        y0=initial_state,
        args=(g, length, damping),
        t_eval=t_eval,
        method='RK45',
        rtol=1e-6,
        atol=1e-8
    )

    # Return both raw solution objects containing time arrays (.t) and state matrices (.y)
    return sol_nl, sol_lin

We now define the function **plot_pendulum_results**. As seen below, the plot function's job is to render the solution to the problem you are working on in the most beautiful way. This sometimes makes the plot function. The key details here are **.plot** and **.scatter** functions that do the heavy lifting.

In [5]:
def plot_pendulum_results(sol_nl, sol_lin, compare_linear=True):
    """
    Renders visual analysis graphs: Time Series theta(t) and Phase Space (omega vs theta).
    
    Parameters:
    ----------
    sol_nl : OdeResult
        SciPy numerical solution object for non-linear pendulum.
    sol_lin : OdeResult
        SciPy numerical solution object for linear pendulum.
    compare_linear : bool
        Toggle to show/hide the linear approximation curves.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)
    
    # Extract starting state coordinates for phase plot origin marker
    initial_theta_rad = sol_nl.y[0][0]
    initial_omega = sol_nl.y[1][0]

    # --- PANEL 1: TIME SERIES GRAPH ---
    axes[0].plot(sol_nl.t, np.degrees(sol_nl.y[0]), color='C0', lw=2, label=r"Nonlinear: $\sin(\theta)$")
    if compare_linear:
        axes[0].plot(sol_lin.t, np.degrees(sol_lin.y[0]), color='C3', linestyle='--', lw=1.5, label=r"Linear Approx: $\theta$")

    axes[0].set(title=r"Angle Trajectory $\theta(t)$", xlabel="Time $t$ (seconds)", ylabel=r"Angle $\theta$ (degrees)")

    # --- PANEL 2: PHASE SPACE PORTRAIT ---
    axes[1].plot(sol_nl.y[0], sol_nl.y[1], color='C0', lw=2, label="Nonlinear Trajectory")
    if compare_linear:
        axes[1].plot(sol_lin.y[0], sol_lin.y[1], color='C3', linestyle='--', lw=1.5, label="Linear Trajectory")

    axes[1].scatter([initial_theta_rad], [initial_omega], color='black', zorder=5, s=40, label="Start State $(t=0)$")
    axes[1].set(title=r"Phase Portrait ($\omega$ vs $\theta$)", xlabel=r"Angle $\theta$ (radians)", ylabel=r"Angular Velocity $\omega$ (rad/s)")

    # Apply shared formatting across both subplots
    for ax in axes:
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(loc="upper right", framealpha=0.9)

    plt.tight_layout()
    plt.show()

## Commenting Exercise: 
We will now practice commenting the code. Please add comments describing the function.

In [6]:
def create_interactive_pendulum_app():
    """
    COMPLETE THIS COMMENTING EXERCISE






    """
    def update_pipeline(theta_deg, omega0, length, damping, compare_linear):
        """

        COMPLETE THIS COMMENTING EXERCISE






        """
    # 1. Call numerical solver
        sol_nl, sol_lin = solve_pendulum_systems(
            theta_deg=theta_deg,
            omega0=omega0,
            length=length,
            damping=damping
        )
        # 2. Render plots
        plot_pendulum_results(sol_nl, sol_lin, compare_linear=compare_linear)

    # Interactive Widgets
    slider_theta = FloatSlider(min=5.0, max=175.0, step=5.0, value=30.0, description=r"Initial $\theta_0$ (°)")
    slider_omega = FloatSlider(min=-6.0, max=6.0, step=0.2, value=0.0, description=r"Initial $\omega_0$")
    slider_length = FloatSlider(min=0.2, max=3.0, step=0.1, value=1.0, description="Length $L$ (m)")
    slider_damping = FloatSlider(min=0.0, max=0.5, step=0.01, value=0.0, description=r"Damping $\gamma$")
    checkbox_linear = Checkbox(value=True, description="Overlay Linear Model")

    # UI Layout Box
    ui_controls = VBox([
        HBox([slider_theta, slider_omega]),
        HBox([slider_length, slider_damping]),
        HBox([checkbox_linear])
    ])

    # Connect UI controls to pipeline
    interactive_output = widgets.interactive_output(
        update_pipeline,
        {
            'theta_deg': slider_theta,
            'omega0': slider_omega,
            'length': slider_length,
            'damping': slider_damping,
            'compare_linear': checkbox_linear
        }
    )

    display(ui_controls, interactive_output)

The main part of the code that calls all the necessary functions.

In [7]:
create_interactive_pendulum_app()

Output()

## Student Exploration Worksheet

### Part 1: Small-Angle Limits
1. Keep **Damping ($\gamma$)** at `0.0` and **Initial Angle** at `10°`. Does the linear model match the exact non-linear model?
2. Increase the **Initial Angle** to `45°`, `90°`, and `160°`.
   * How does the period of the true pendulum change compared to the constant period predicted by the linear model?
   * Why does the non-linear curve take on a non-sinusoidal shape at large angles?

### Part 2: Energy & Phase Space Topography
1. Set **Damping ($\gamma$)** to `0.0` and **Initial Angle** to `60°`. Describe the geometric shape formed in the phase space plot.
2. Slowly increase **Initial Speed ($\omega_0$)** from `0.0` to `5.0`. At what point does the closed loop in phase space break open into an oscillating wave? What physical movement of the pendulum does this topological change represent?